# Tiny Text-to-Cypher Fine-Tuning with SmolLM2-360M

This notebook is the end-to-end Colab workflow for the project.

What you will learn as you run it:
- how to shape a text-to-Cypher dataset
- why synthetic schema-grounded data matters
- how to benchmark the untuned base model before training
- how QLoRA fine-tuning works for a tiny instruct model
- how to evaluate outputs with execution-based checks instead of string match alone

The v1 goal is simple: make `HuggingFaceTB/SmolLM2-360M-Instruct` meaningfully better at Cypher generation.


## 1. Environment Setup

Why this step matters:
- Colab runtimes are ephemeral, so every serious notebook should install exact dependencies up front.
- `pip install -e .` keeps the reusable package and the notebook in sync.
- `nvidia-smi` tells you whether Colab gave you T4, L4, A100, or something else.
- this project intentionally teaches only the CUDA + QLoRA path used in most production fine-tuning workflows
- if the repo is not already present in Colab, this setup cell will clone it automatically from GitHub
- the current training defaults are tuned for an NVIDIA A100 in Colab


In [1]:
%pip install -q -U pip setuptools wheel

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/spabolu/text-to-cypher.git"
REPO_DIR = "text-to-cypher"

repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "pyproject.toml").exists() and (candidate / "cypher_slm").exists():
        repo_root = candidate
        break

if repo_root is None:
    clone_target = Path.cwd() / REPO_DIR
    if not clone_target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_target)], check=True)
    repo_root = clone_target

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(repo_root)
%pip install -q -e {repo_root}

print(f"Using repo root: {repo_root}")
os.system("nvidia-smi")


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cypher-slm (pyproject.toml) ... done
Using repo root: /content/text-to-cypher


0

## 2. Imports and Run Configuration

What to inspect here:
- artifact directories
- base model name
- training defaults like learning rate, sequence length, and LoRA rank

You can treat this as the control panel for the rest of the notebook.

In [2]:
import importlib.util
import os
import sys

import pandas as pd
from IPython.display import display

assert importlib.util.find_spec("cypher_slm") is not None, (
    "cypher_slm is still not importable. Make sure you cloned or uploaded the full repo, "
    "then rerun the setup cell."
)

from cypher_slm.config import RunConfig
from cypher_slm.data import build_training_corpus, load_public_examples, save_jsonl
from cypher_slm.evaluation import evaluate_examples, save_evaluation_records
from cypher_slm.prompts import SYSTEM_PROMPT, render_user_prompt
from cypher_slm.reporting import summarize_records, write_markdown_report
from cypher_slm.synthetic import build_demo_schemas, generate_synthetic_examples
from cypher_slm.training import (
    build_generation_pipeline,
    export_training_config,
    train_qlora,
    generate_query,
)

run_config = RunConfig()
run_config.artifacts.ensure()

print("A100-tuned training config:")
print(run_config.training)


TrainingConfig(base_model='HuggingFaceTB/SmolLM2-360M-Instruct', output_dir='artifacts/models/smollm2-360m-cypher', max_length=768, learning_rate=0.0002, num_train_epochs=3.0, per_device_train_batch_size=8, per_device_eval_batch_size=8, gradient_accumulation_steps=2, warmup_ratio=0.05, weight_decay=0.01, logging_steps=10, eval_steps=50, save_steps=50, max_steps=-1, lora_r=32, lora_alpha=64, lora_dropout=0.05, seed=42)


## 3. Build the Training Corpus

Industry-standard lesson:
- never start training before you know exactly where the data came from
- keep public data, synthetic data, and benchmark-only data conceptually separate
- reserve at least one schema for held-out evaluation to test generalization

In this v1 pipeline:
- public Hugging Face text-to-Cypher examples are loaded if available
- synthetic schema-grounded examples are always generated
- the `commerce` schema is held out for test-only evaluation by default

In [3]:
demo_schemas = build_demo_schemas()
public_examples = load_public_examples()
synthetic_examples = generate_synthetic_examples(demo_schemas)
corpus = build_training_corpus(
    public_examples=public_examples,
    synthetic_examples=synthetic_examples,
    held_out_schema_ids=["commerce"],
    validation_fraction=0.15,
)

train_path = save_jsonl(corpus, run_config.artifacts.processed_data / "training_corpus.jsonl")

rows = [example.to_dict() for example in corpus]
df = pd.DataFrame(rows)
print(f"Saved corpus to {train_path}")
print(f"Total rows: {len(df)}")
display(df.groupby(["split", "source"]).size().reset_index(name="rows"))
display(df.groupby(["schema_id", "split"]).size().reset_index(name="rows"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


dataset-trng_000.jsonl: 0.00B [00:00, ?B/s]

dataset-trng_100.jsonl: 0.00B [00:00, ?B/s]

dataset-trng_200.jsonl: 0.00B [00:00, ?B/s]

dataset-trng_300.jsonl: 0.00B [00:00, ?B/s]

...

text_to_cypher.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/149 [00:00<?, ? examples/s]

Saved corpus to artifacts/processed/training_corpus.jsonl
Total rows: 38


,split,source,rows
0,test,synthetic/template:contains,2
1,test,synthetic/template:count-nodes,4
2,test,synthetic/template:in_category,2
3,test,synthetic/template:list-properties,4
4,test,synthetic/template:placed,2
5,train,synthetic/template:acted_in,2
6,train,synthetic/template:count-nodes,4
7,train,synthetic/template:directed,2
8,train,synthetic/template:follows,2
9,train,synthetic/template:in_genre,1


,schema_id,split,rows
0,commerce,test,14
1,movies,train,10
2,movies,validation,2
3,social,train,10
4,social,validation,2


## 4. Inspect the Prompt Format

Why this matters:
- the model only learns the task you actually present to it
- prompt formatting is part of the training recipe, not an afterthought
- we want the assistant output to be raw Cypher only, because prose hurts both training signal and evaluation discipline

In [4]:
sample = corpus[0]
print("SYSTEM PROMPT:\n")
print(SYSTEM_PROMPT)
print("\nUSER PROMPT:\n")
print(render_user_prompt(sample.schema_text, sample.question))
print("\nTARGET CYPHER:\n")
print(sample.cypher)

SYSTEM PROMPT:

You are an expert Neo4j engineer.
Generate a valid Cypher query for the given graph schema and user request.
Return only raw Cypher.
Do not add markdown fences, commentary, or explanations.

USER PROMPT:

Graph schema:
        A movie recommendation graph with people, movies, genres, and ratings.
Node Person { name: string, born: integer }
Node Movie { title: string, released: integer, tagline: string }
Node Genre { name: string }
Relationship (: Person)-[:ACTED_IN { roles: list[string] }]->(: Movie)
Relationship (: Person)-[:DIRECTED { no properties }]->(: Movie)
Relationship (: Movie)-[:IN_GENRE { no properties }]->(: Genre)

        Task:
        Write a Cypher query that answers the following question.

        Question: List the name values for all person nodes.

TARGET CYPHER:

MATCH (n:Person) RETURN n.name AS name


## 5. Baseline the Untuned Model Before Training

This is a critical habit.

If you do not benchmark the base model first, you cannot later prove that fine-tuning helped.
The first pass here is qualitative: look at a few generations and see whether the model follows the output format.

In [6]:
base_generator = build_generation_pipeline(run_config.training.base_model)

baseline_preview = []
for example in corpus[:3]:
    generated = generate_query(base_generator, example.schema_text, example.question)
    baseline_preview.append(
        {
            "schema_id": example.schema_id,
            "question": example.question,
            "expected": example.cypher,
            "generated": generated,
        }
    )

display(pd.DataFrame(baseline_preview))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,schema_id,question,expected,generated
0,movies,List the name values for all person nodes.,MATCH (n:Person) RETURN n.name AS name,Cypher query:\n\n```\nSELECT name\nFROM Person...
1,movies,How many person nodes are in the graph?,MATCH (n:Person) RETURN count(n) AS personCount,Cypher query:\n\n```\nSELECT COUNT(*)\nFROM (:...
2,movies,List the title values for all movie nodes.,MATCH (n:Movie) RETURN n.title AS title,Cypher query:\n\n```\nMATCH (n:Person)-[:ACTED...


## 6. Optional Execution-Based Evaluation Against Neo4j

Why execution-based evaluation is better than exact string match:
- multiple Cypher queries can be semantically equivalent
- formatting differences should not count as failures
- the real question is whether the query runs and returns the right result

To enable this section, provide Neo4j credentials as Colab secrets or environment variables:
- `NEO4J_URI`
- `NEO4J_USERNAME`
- `NEO4J_PASSWORD`
- optional `NEO4J_DATABASE`

In [7]:
neo4j_uri = os.getenv("NEO4J_URI", run_config.neo4j_uri)
neo4j_username = os.getenv("NEO4J_USERNAME", run_config.neo4j_username)
neo4j_password = os.getenv("NEO4J_PASSWORD", "")
neo4j_database = os.getenv("NEO4J_DATABASE", run_config.neo4j_database or "") or None

if neo4j_password:
    baseline_records = evaluate_examples(
        examples=[example for example in corpus if example.split == "test"],
        model_id=run_config.training.base_model,
        generator=lambda schema_text, question: generate_query(base_generator, schema_text, question),
        neo4j_uri=neo4j_uri,
        neo4j_username=neo4j_username,
        neo4j_password=neo4j_password,
        database=neo4j_database,
    )
    baseline_eval_path = save_evaluation_records(
        baseline_records,
        run_config.artifacts.reports / "baseline_eval.jsonl",
    )
    baseline_summary = summarize_records(baseline_records)
    display(baseline_summary)
    print(f"Saved baseline evaluation to {baseline_eval_path}")
else:
    print("Skipping execution-based baseline evaluation because NEO4J_PASSWORD is not set.")

Skipping execution-based baseline evaluation because NEO4J_PASSWORD is not set.


## 7. Fine-Tune with QLoRA

What you are learning here:
- full fine-tuning updates every weight, which is unnecessary for a small specialization task
- LoRA trains a small adapter instead of the full model
- QLoRA adds 4-bit quantization so the base model consumes less memory while the adapters still learn effectively

Suggested workflow:
- first run a smoke experiment by lowering `num_train_epochs` or limiting the dataset
- once the pipeline works, restore the full config and rerun

This project intentionally supports only the CUDA + QLoRA path so the training recipe stays aligned with standard NVIDIA-based fine-tuning workflows.
The defaults in this notebook assume an NVIDIA A100, so they are less conservative than T4-style settings.


In [8]:
export_training_config(
    run_config.training,
    run_config.artifacts.model_outputs / "smollm2_training_config.txt",
)

trainer = train_qlora(corpus, run_config.training)
print(f"Model artifacts saved under {run_config.training.output_dir}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss


Model artifacts saved under artifacts/models/smollm2-360m-cypher


## 8. Evaluate the Tuned Model

This section mirrors the baseline step.
The only honest comparison is to keep the benchmark constant and change only the model.

In [9]:
tuned_generator = build_generation_pipeline(run_config.training.output_dir)

tuned_preview = []
for example in corpus[:3]:
    generated = generate_query(tuned_generator, example.schema_text, example.question)
    tuned_preview.append(
        {
            "schema_id": example.schema_id,
            "question": example.question,
            "expected": example.cypher,
            "generated": generated,
        }
    )

display(pd.DataFrame(tuned_preview))

if neo4j_password:
    tuned_records = evaluate_examples(
        examples=[example for example in corpus if example.split == "test"],
        model_id=run_config.training.output_dir,
        generator=lambda schema_text, question: generate_query(tuned_generator, schema_text, question),
        neo4j_uri=neo4j_uri,
        neo4j_username=neo4j_username,
        neo4j_password=neo4j_password,
        database=neo4j_database,
    )
    tuned_eval_path = save_evaluation_records(
        tuned_records,
        run_config.artifacts.reports / "tuned_eval.jsonl",
    )
    tuned_summary = summarize_records(tuned_records)
    display(tuned_summary)
    print(f"Saved tuned evaluation to {tuned_eval_path}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,schema_id,question,expected,generated
0,movies,List the name values for all person nodes.,MATCH (n:Person) RETURN n.name AS name,MATCH (n:Person) RETURN n.name AS name
1,movies,How many person nodes are in the graph?,MATCH (n:Person) RETURN count(n) AS personCount,MATCH (n:Person)-[:ACTED_IN]->(m:Movie) RETURN...
2,movies,List the title values for all movie nodes.,MATCH (n:Movie) RETURN n.title AS title,MATCH (n:Person)-[:ACTED_IN]->(m:Movie)\nRETUR...


## 9. Export a Simple Benchmark Report

The final artifact should be understandable without opening the notebook.
That is why we export the summary table to markdown.

In [10]:
if neo4j_password:
    report_path = write_markdown_report(
        tuned_summary,
        run_config.artifacts.reports / "benchmark_summary.md",
        title="SmolLM2-360M Text-to-Cypher Benchmark Summary",
    )
    print(f"Wrote markdown summary to {report_path}")
else:
    print("Skipping report generation because no execution-based summary exists yet.")

Skipping report generation because no execution-based summary exists yet.


## Next Learning Iterations

Once this notebook works end-to-end, the next useful experiments are:
1. change the held-out schema and measure generalization shift
2. ablate synthetic-only vs public-plus-synthetic data
3. rerun the same pipeline on `meta-llama/Llama-3.2-1B-Instruct`
4. compare adapter-only export vs merged model export for inference convenience